# Chapter 2: one question drawn three ways

The question is "is tank F07 behaving?". The twelve Kestrel pale ale fills of chapter 5 are drawn three ways:
by calendar date, by batch hour and as small multiples, one panel per tank. Look at which view answers the question
in the fewest seconds. Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

In [ ]:
fill = load("Fill", parse_dates=["HourZero"])
hg = load("HourlyGravity").merge(fill[["FillId", "Tank", "HourZero", "IsReference"]], on="FillId")
hg["Time"] = hg.HourZero + pd.to_timedelta(hg.BatchHour, unit="h")
f07 = hg[hg.Tank == "F07"]
print(f07.FillId.iloc[0], "is the F07 fill;", hg.FillId.nunique(), "fills in all")

## View A: calendar date
Every fill starts on a different day, so the curves smear across the page.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for fid, g in hg.groupby("FillId"):
    ax.plot(g.Time, g.Gravity, color="0.75", lw=1)
ax.plot(f07.Time, f07.Gravity, color="tab:blue", lw=2, label="F07")
ax.set(title="Gravity by calendar date, twelve pale ale fills", ylabel="Gravity (degrees Plato)")
ax.legend(); fig.autofmt_xdate(); plt.show()

## View B: batch hour (the one marked best in the book)
Hour zero is the end of pitching, stored on the fill. Every curve now starts in the same place.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for fid, g in hg.groupby("FillId"):
    ax.plot(g.BatchHour, g.Gravity, color="0.75", lw=1)
ax.plot(f07.BatchHour, f07.Gravity, color="tab:blue", lw=2, label="F07")
ax.set(title="Gravity by batch hour, twelve pale ale fills", xlabel="Batch hour", ylabel="Gravity (degrees Plato)")
ax.legend(); plt.show()

## View C: small multiples, shared axes
One panel per tank on the same axes, ordered by tank number. The second library here is pandas plotting,
which draws the same panels with less code.

In [ ]:
wide = hg.pivot_table(index="BatchHour", columns="Tank", values="Gravity")
axes = wide.plot(subplots=True, layout=(3, 4), figsize=(10, 6), sharex=True, sharey=True, legend=False, color="0.4")
for ax, tank in zip(axes.flat, wide.columns):
    ax.set_title(tank, fontsize=9)
plt.suptitle("Gravity by batch hour, one panel per tank, degrees Plato"); plt.tight_layout(); plt.show()

**What to look at.** F07 sits inside the bundle in view B. The flat line in view C belongs to F09, the stalled fill.
To try it on your own data, swap `HourlyGravity.csv` for an export with the same three columns.